In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT,WORK=Path('/kaggle/input'),Path('/kaggle/working')
def newest(pattern,label):
    paths=list(INPUT.rglob(pattern))
    if not paths: raise FileNotFoundError(f'Attach saved output containing {label}: {pattern}')
    return max(paths,key=lambda p:p.stat().st_mtime)

binary_sources=[]
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name!='binary': continue
    try: meta=json.loads(mp.read_text(encoding='utf-8'))
    except Exception: continue
    if meta.get('family')=='binary' and int(meta.get('parameter_draws_unique',0))>=100_000:
        binary_sources.append((meta.get('created_utc',''),mp.parent))
if not binary_sources: raise FileNotFoundError('Attach Day 7 Cosmic swan Bayesian binary simulations.')
_,BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
parameters=pd.read_csv(BINARY_DIR/'parameters.csv')

MAP_PATH=newest('day9_12_conditional_mle_parameters.csv','Day 9.12 conditional MLE estimates')
NUTS_CASE_PATH=newest('day9_13_nuts_case_diagnostics.csv','Day 9.13 case diagnostics')
NUTS_PARAM_PATH=newest('day9_13_per_parameter_diagnostics.csv','Day 9.13 per-parameter diagnostics')
case_df=pd.read_csv(NUTS_CASE_PATH)
param_diag=pd.read_csv(NUTS_PARAM_PATH)
map_df=pd.read_csv(MAP_PATH)
if not {'simulation_row','status','period_fixed_profile_min_days'}.issubset(case_df.columns):
    raise ValueError('Day 9.13 case diagnostics do not have expected columns.')
REVIEW_ROWS=sorted(case_df.loc[case_df.status.eq('needs_review'),'simulation_row'].astype(int).tolist())
PASS_ROWS=sorted(case_df.loc[case_df.status.eq('screen_pass'),'simulation_row'].astype(int).tolist())
if not REVIEW_ROWS: raise ValueError('No Day 9.13 needs_review cases found.')
# Include one sampler-pass case as a visual/diagnostic reference where available.
ROWS=REVIEW_ROWS+(PASS_ROWS[:1] if PASS_ROWS else [])
OUT=WORK/('day9-14-posterior-forensics-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)
print('[PASS] Day 7:',BINARY_DIR)
print('[PASS] Day 9.12 MLE estimates:',MAP_PATH.parent)
print('[PASS] Day 9.13 diagnostics:',NUTS_CASE_PATH.parent)
print('Review rows:',REVIEW_ROWS,'| passing comparison:',PASS_ROWS[:1])
print('Output:',OUT)

In [ ]:
latent_names=['baseline_flux','phase_offset','depth_primary','depth_ratio','duration_fraction',
              'ellipsoidal_amplitude','reflection_amplitude','log_noise']
support={
    'depth_primary':(0.002,0.45),
    'depth_ratio':(0.0,1.0),
    'duration_fraction':(0.005,0.12),
    'ellipsoidal_amplitude':(0.0,0.02),
    'reflection_amplitude':(-0.01,0.01),
    'log_noise':(float(np.log(1e-4)),float(np.log(0.03))),
}
summary_rows=[]
plot_samples={}
truths_by_row={}
for row in ROWS:
    npz_path=NUTS_CASE_PATH.parent/f'conditional_row_{row}_posterior_by_chain.npz'
    if not npz_path.exists(): raise FileNotFoundError(f'Missing chain-preserving posterior for row {row}: {npz_path.name}')
    with np.load(npz_path) as archive:
        samples={key:np.asarray(archive[key],dtype=float) for key in archive.files}
    if any(name not in samples for name in latent_names):
        raise ValueError(f'Posterior row {row} is missing one or more expected latent parameters.')
    if samples['baseline_flux'].ndim!=2 or samples['baseline_flux'].shape[0]!=2:
        raise ValueError(f'Expected two chain x draw arrays for row {row}.')
    seed_rows=map_df[map_df.simulation_row.astype(int).eq(row)]
    seed=seed_rows.set_index('parameter').estimate.astype(float).to_dict()
    if 'phase' not in seed: raise ValueError(f'Missing refined phase seed for row {row}.')
    truth=parameters.iloc[row]
    truth_map={name:float(truth[name]) for name in ['phase','depth_primary','depth_ratio','duration_fraction',
        'ellipsoidal_amplitude','reflection_amplitude','noise_sigma']}
    truth_map['baseline_flux']=1.0
    phase0=float(seed['phase']%1.0)
    truth_map['phase_offset']=float((truth_map['phase']-phase0+0.5)%1.0-0.5)
    truth_map['log_noise']=float(np.log(truth_map['noise_sigma']))
    truths_by_row[row]=truth_map
    plot_samples[row]=samples
    for name in latent_names:
        arr=samples[name]
        pooled=arr.reshape(-1)
        chain_means=np.mean(arr,axis=1)
        pooled_sd=float(np.std(pooled,ddof=1)) if pooled.size>1 else float('nan')
        mean_diff_sd=float(abs(chain_means[0]-chain_means[1])/pooled_sd) if pooled_sd>0 else float('nan')
        q05,median,q95=np.quantile(pooled,[0.05,0.5,0.95])
        lo,hi=support.get(name,(None,None))
        if lo is None:
            mle_value=0.0 if name=='phase_offset' else seed.get(name,float('nan'))
            frac_near_lower=frac_near_upper=float('nan')
            normalized_mle_edge_distance=float('nan')
        else:
            width=hi-lo
            frac_near_lower=float(np.mean(pooled-lo < 0.01*width))
            frac_near_upper=float(np.mean(hi-pooled < 0.01*width))
            mle_value=float(seed.get(name,np.nan))
            if name=='log_noise': mle_value=float(np.log(seed['noise_sigma']))
            normalized_mle_edge_distance=float(min(mle_value-lo,hi-mle_value)/width)
        truth_value=truth_map[name]
        summary_rows.append({'simulation_row':row,'status':str(case_df.loc[case_df.simulation_row.astype(int).eq(row),'status'].iloc[0]),
            'parameter':name,'chain1_mean':float(chain_means[0]),'chain2_mean':float(chain_means[1]),
            'chain_mean_difference_in_pooled_sd':mean_diff_sd,'q05':float(q05),'median':float(median),'q95':float(q95),
            'known_truth_evaluation_only':float(truth_value),'truth_inside_90pct_interval':bool(q05<=truth_value<=q95),
            'mle_start':float(mle_value),'fraction_samples_near_lower_1pct':frac_near_lower,
            'fraction_samples_near_upper_1pct':frac_near_upper,'mle_normalized_distance_to_nearest_bound':normalized_mle_edge_distance})

summary_df=pd.DataFrame(summary_rows)
summary_df.to_csv(OUT/'day9_14_parameter_forensics.csv',index=False)
param_diag.to_csv(OUT/'day9_14_sampler_diagnostics.csv',index=False)

nrows=len(ROWS)
fig,axes=plt.subplots(nrows,len(latent_names),figsize=(20,3.0*nrows),squeeze=False,constrained_layout=True)
for ri,row in enumerate(ROWS):
    samples=plot_samples[row]
    for ci,name in enumerate(latent_names):
        ax=axes[ri,ci]
        arr=samples[name]
        for chain in range(arr.shape[0]):
            ax.plot(np.arange(arr.shape[1]),arr[chain],lw=0.55,alpha=0.8,label=f'chain {chain+1}')
        truth_val=truths_by_row[row][name]
        ax.axhline(truth_val,color='crimson',lw=0.8,ls='--',label='known truth')
        ax.set_title(f'row {row}: {name}',fontsize=8)
        if ri==nrows-1: ax.set_xlabel('draw')
        if ci==0: ax.set_ylabel('sample value')
        if ri==0 and ci==0: ax.legend(fontsize=7)
fig.savefig(OUT/'day9_14_chain_traces.png',dpi=140)
plt.show()

boundary=summary_df[summary_df.parameter.isin(support)].copy()
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'day9_13_input':str(NUTS_CASE_PATH.parent),
    'review_rows':REVIEW_ROWS,'passing_comparison_rows':PASS_ROWS[:1],
    'diagnostic_purpose':'Inspect conditional NUTS chain behavior, selected synthetic truths, and proximity to parameter support bounds.',
    'caution':'A chain near a bound is a diagnostic clue only; do not change model support without checking simulation design and scientific assumptions.',
    'outputs':['day9_14_parameter_forensics.csv','day9_14_sampler_diagnostics.csv','day9_14_chain_traces.png']}
(OUT/'day9_14_posterior_forensics_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Case-level Day 9.13 diagnostics:')
print(case_df.to_string(index=False))
print('Parameter forensics:')
print(summary_df.to_string(index=False))
print(json.dumps(summary,indent=2))
print('Outputs:',OUT)